# Ejemplo de Observables y Animación 1D

Este notebook demuestra cómo:
1. Configurar y ejecutar una simulación 1D de un paquete de ondas gaussiano.
2. Calcular y guardar observables físicos como la posición, momento y energía esperados.
3. Generar una animación que muestre la evolución de la densidad de probabilidad y la trayectoria de la posición esperada.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
from dirac_solver.solver import DiracSolver
from dirac_solver.mesher import Mesher
from dirac_solver.initial_state import GaussianPacket
from dirac_solver.potentials import FreeParticle
from dirac_solver.storage import HDF5Storage, create_animation_1d
from dirac_solver.observables import Observables

## 1. Configuración de la Simulación

In [ ]:
# --- Parámetros de la simulación ---
grid = Mesher.create_grid(shape=(200,), spacing=(0.1,))
time_step = 0.01
total_time = 10.0

# --- Estado Inicial: Paquete Gaussiano ---
initial_state = GaussianPacket(
    grid,
    position_mean=np.array([0.0]),
    momentum_mean=np.array([0.5]),
    width=2.0
)

# --- Potencial y Condiciones de Borde ---
potential = FreeParticle()
boundary_condition = "periodic"

# --- Construcción del Problema ---
problem = (
    DiracSolver.builder()
    .set_grid(grid)
    .set_initial_state(initial_state)
    .set_potential(potential)
    .set_boundary_condition(boundary_condition)
    .set_time_parameters(time_step, total_time)
    .build()
)

solver = DiracSolver(problem)

## 2. Ejecución de la Simulación y Cálculo de Observables

In [ ]:
# --- Archivos de salida ---
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)
hdf5_path = os.path.join(output_dir, 'simulation_1d.hdf5')
animation_path = os.path.join(output_dir, 'simulation_1d.gif')

# --- Inicializar manejadores ---
storage = HDF5Storage(hdf5_path, problem)
observables = Observables(solver)

# --- Ejecutar la simulación ---
solver.run_simulation(
    storage_handler=storage,
    observables_handler=observables,
    save_every_n_steps=5
)

## 3. Visualización de los Observables

In [ ]:
time = observables.get_history('time')
expected_pos = observables.get_history('expected_position')
expected_mom = observables.get_history('expected_momentum')
expected_energy = observables.get_history('expected_energy')

plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.plot(time, expected_pos[:, 0])
plt.ylabel('Posición Esperada <x>')
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(time, expected_mom[:, 0])
plt.ylabel('Momento Esperado <px>')
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(time, expected_energy)
plt.ylabel('Energía Esperada <E>')
plt.xlabel('Tiempo')
plt.grid(True)

plt.suptitle('Evolución de los Observables')
plt.show()

## 4. Generación de la Animación

In [ ]:
print(f"Generando animación a partir de {hdf5_path}...")
create_animation_1d(hdf5_path, animation_path, interval=50)
print(f"Animación guardada en: {animation_path}")